# 资源和模板
向您的 MCP 客户端公开数据源和动态内容生成器。

资源表示 MCP 客户端可以读取的数据或文件，资源模板通过允许客户端根据 URI 中传递的参数请求动态生成的资源来扩展此概念。

FastMCP 主要使用`@mcp.resource`装饰器简化了静态和动态资源的定义。

什么是资源？
资源为 LLM 或客户端应用程序提供对数据的只读访问权限。当客户端请求资源 URI 时：

- FastMCP 找到相应的资源定义。
- 如果它是动态的（由函数定义），则执行该函数。
- 将内容（文本、JSON、二进制数据）返回给客户端。
- 这允许 LLM 访问与对话相关的文件、数据库内容、配置或动态生成的信息。

## 资源
​`@resource装饰者`           
定义资源最常见的方式是装饰一个 Python 函数。装饰器需要资源的唯一 URI。

In [2]:
import json
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")

# Basic dynamic resource returning a string
@mcp.resource("resource://greeting")
def get_greeting() -> str:
    """Provides a simple greeting message."""
    return "Hello from FastMCP Resources!"

# Resource returning JSON data (dict is auto-serialized)
@mcp.resource("data://config")
def get_config() -> dict:
    """Provides application configuration as JSON."""
    return {
        "theme": "dark",
        "version": "1.2.0",
        "features": ["tools", "resources"],
    }

ModuleNotFoundError: No module named 'uvicorn'

 关键概念：

- URI：@resource的第一个参数是客户端用来请求该数据的唯一URI（例如"resource://greeting"）。
- 延迟加载：仅当客户端通过 sources/read 专门请求该资源 URI 时，才会执行装饰函数（get_greeting， get_configre） 。
- 推断元数据：默认情况下：
- 资源名称：取自函数名称（get_greeting）。
- 资源描述：取自函数的文档字符串。
​


### 返回值
FastMCP 自动将你的函数的返回值转换为适当的 MCP 资源内容：

- str：发送方式TextResourceContents（mime_type="text/plain"默认）。
- dict，，list：pydantic.BaseModel自动序列化为 JSON 字符串并发送TextResourceContents（mime_type="application/json"默认为）。
- bytes：Base64 编码并发送为BlobResourceContents。您应该指定适当的编码mime_type（例如"image/png"、"application/octet-stream"）。
- None：导致返回空的资源内容列表。

### 资源元数据
您可以使用装饰器中的参数自定义资源的属性：

- uri：资源的唯一标识符（必需）。
- name：人类可读的名称（默认为函数名称）。
- description：资源的解释（默认为文档字符串）。
- mime_type：指定内容类型（FastMCP 通常会推断出默认值，text/plain如 或application/json，但对于非文本类型，明确表示更好）。
- tags：一组用于分类的字符串，可能由客户端用于过滤。

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")

# Example specifying metadata
@mcp.resource(
    uri="data://app-status",      # Explicit URI (required)
    name="ApplicationStatus",     # Custom name
    description="Provides the current status of the application.", # Custom description
    mime_type="application/json", # Explicit MIME type
    tags={"monitoring", "status"} # Categorization tags
)
def get_application_status() -> dict:
    """Internal function description (ignored if description is provided above)."""
    return {"status": "ok", "uptime": 12345, "version": mcp.settings.version} # Example usage

### 访问 MCP 上下文

资源和资源模板可以通过该`Context`对象访问额外的 MCP 信息和功能。要访问它，请向资源函数添加一个参数，并在其类型注释中注明`Context`


In [ ]:
from fastmcp import FastMCP, Context

mcp = FastMCP(name="DataServer")

@mcp.resource("resource://system-status")
async def get_system_status(ctx: Context) -> dict:
    """Provides system status information."""
    return {
        "status": "operational",
        "request_id": ctx.request_id
    }

@mcp.resource("resource://{name}/details")
async def get_details(name: str, ctx: Context) -> dict:
    """Get details for a specific name."""
    return {
        "name": name,
        "accessed_at": ctx.request_id
    }

### 异步资源
用于async def执行 I/O 操作（例如从数据库或网络读取）的资源功能，以避免阻塞服务器。

In [ ]:
import aiofiles
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")

@mcp.resource("file:///app/data/important_log.txt", mime_type="text/plain")
async def read_important_log() -> str:
    """Reads content from a specific log file asynchronously."""
    try:
        async with aiofiles.open("/app/data/important_log.txt", mode="r") as f:
            content = await f.read()
        return content
    except FileNotFoundError:
        return "Log file not found."

### 资源类
虽然 `@mcp.resource` 非常适合动态内容，但您也可以使用 `mcp.add_resource()` 和具体的 `Resource` 子类直接注册预定义的资源（如静态文件或简单文本）。

In [ ]:
from pathlib import Path
from fastmcp import FastMCP
from fastmcp.resources import FileResource, TextResource, DirectoryResource

mcp = FastMCP(name="DataServer")

# 1. Exposing a static file directly
readme_path = Path("./README.md").resolve()
if readme_path.exists():
    # Use a file:// URI scheme
    readme_resource = FileResource(
        uri=f"file://{readme_path.as_posix()}",
        path=readme_path, # Path to the actual file
        name="README File",
        description="The project's README.",
        mime_type="text/markdown",
        tags={"documentation"}
    )
    mcp.add_resource(readme_resource)

# 2. Exposing simple, predefined text
notice_resource = TextResource(
    uri="resource://notice",
    name="Important Notice",
    text="System maintenance scheduled for Sunday.",
    tags={"notification"}
)
mcp.add_resource(notice_resource)

# 3. Using a custom key different from the URI
special_resource = TextResource(
    uri="resource://common-notice",
    name="Special Notice",
    text="This is a special notice with a custom storage key.",
)
mcp.add_resource(special_resource, key="resource://custom-key")

# 4. Exposing a directory listing
data_dir_path = Path("./app_data").resolve()
if data_dir_path.is_dir():
    data_listing_resource = DirectoryResource(
        uri="resource://data-files",
        path=data_dir_path, # Path to the directory
        name="Data Directory Listing",
        description="Lists files available in the data directory.",
        recursive=False # Set to True to list subdirectories
    )
    mcp.add_resource(data_listing_resource) # Returns JSON list of files

常见资源类别：

- `TextResource`：用于简单的字符串内容。
- `BinaryResource`：针对原始bytes内容。
- `FileResourc`e：从本地文件路径读取内容。处理文本/二进制模式和惰性读取。
- `HttpResource`：从 HTTP(S) URL 获取内容（需要httpx）。
- `DirectoryResource`：列出本地目录中的文件（返回 JSON）。
- （`FunctionResource`：使用的内部类@mcp.resource）。 
当内容是静态的或直接来自文件/URL 时使用这些，从而无需专用的 Python 函数。

### 自定义资源键
直接使用 `mcp.add_resource()`添加资源时，您可以选择提供自定义存储键：

> 请注意，此参数仅在`add_resource()`直接使用时可用，而不是通过`@resource`装饰器，因为在使用装饰器时会明确提供 URI。

In [ ]:
# Creating a resource with standard URI as the key
resource = TextResource(uri="resource://data")
mcp.add_resource(resource)  # Will be stored and accessed using "resource://data"

# Creating a resource with a custom key
special_resource = TextResource(uri="resource://special-data")
mcp.add_resource(special_resource, key="internal://data-v2")  # Will be stored and accessed using "internal://data-v2"

## 资源模板

资源模板允许客户端请求其内容取决于 URI 中嵌入的参数的资源。使用相同的`@mcp.resource`装饰器定义模板，但在 URI 字符串中包含`{parameter_name}`占位符，并在函数签名中添加相应的参数。

资源模板与常规资源共享大多数配置选项（`name`, `description`, `mime_type`, `tags`），但增加了定义映射到函数参数的 URI 参数的能力。

资源模板会为每组独特的参数生成一个新资源，这意味着资源可以按需动态创建。例如，如果资源模板`"user://profile/{name}"`已注册，MCP 客户端可以请求`"user://profile/ford"`或`"user://profile/marvin"`检索这两个用户配置文件中的任意一个作为资源，而无需单独注册每个资源。

> 带有函数的函数*args不支持作为资源模板。但是，与工具和提示不同，资源模板支持，**kwargs因为 URI 模板定义了将被收集并作为关键字参数传递的特定参数名称。

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")

# Template URI includes {city} placeholder
@mcp.resource("weather://{city}/current")
def get_weather(city: str) -> dict:
    """Provides weather information for a specific city."""
    # In a real implementation, this would call a weather API
    # Here we're using simplified logic for example purposes
    return {
        "city": city.capitalize(),
        "temperature": 22,
        "condition": "Sunny",
        "unit": "celsius"
    }

# Template with multiple parameters
@mcp.resource("repos://{owner}/{repo}/info")
def get_repo_info(owner: str, repo: str) -> dict:
    """Retrieves information about a GitHub repository."""
    # In a real implementation, this would call the GitHub API
    return {
        "owner": owner,
        "name": repo,
        "full_name": f"{owner}/{repo}",
        "stars": 120,
        "forks": 48
    }

定义这两个模板后，客户端可以请求各种资源：

- `weather://london/current`→ 返回伦敦的天气
- `weather://paris/current`→ 返回巴黎的天气
- `repos://jlowin/fastmcp/info`→ 返回有关 jlowin/fastmcp 存储库的信息
- `repos://prefecthq/prefect/info`→ 返回有关 prefecthq/prefect 存储库的信息

### 通配符参数

资源模板支持通配符参数，可以匹配多个路径段。标准参数 ( `{param}`) 仅匹配单个路径段，且不会跨越“/”边界，而通配符参数 ( `{param*}`) 可以捕获多个路径段，包括斜杠。通配符会捕获所有后续路径段，直到URI 模板的定义部分（无论是文字还是其他参数）。这允许您在单个 URI 模板中使用多个通配符参数。

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")


# Standard parameter only matches one segment
@mcp.resource("files://{filename}")
def get_file(filename: str) -> str:
    """Retrieves a file by name."""
    # Will only match files://<single-segment>
    return f"File content for: {filename}"


# Wildcard parameter can match multiple segments
@mcp.resource("path://{filepath*}")
def get_path_content(filepath: str) -> str:
    """Retrieves content at a specific path."""
    # Can match path://docs/server/resources.mdx
    return f"Content at path: {filepath}"


# Mixing standard and wildcard parameters
@mcp.resource("repo://{owner}/{path*}/template.py")
def get_template_file(owner: str, path: str) -> dict:
    """Retrieves a file from a specific repository and path, but 
    only if the resource ends with `template.py`"""
    # Can match repo://jlowin/fastmcp/src/resources/template.py
    return {
        "owner": owner,
        "path": path + "/template.py",
        "content": f"File at {path}/template.py in {owner}'s repository"
    }

通配符参数在以下情况下很有用：

- 使用文件路径或分层数据
- 创建需要捕获可变长度路径段的 API
- 构建类似于 REST API 的 URL 模式

请注意，与常规参数一样，每个通配符参数在函数签名中仍必须是命名参数，并且所有必需的函数参数都必须出现在 URI 模板中。

## 默认值

在创建资源模板时，FastMCP 对 URI 模板参数和函数参数之间的关系强制执行两个规则：

- 必需函数参数：所有没有默认值的函数参数（必需参数）都必须出现在 URI 模板中。
- URI 参数：所有 URI 模板参数都必须作为函数参数存在。


但是，具有默认值的函数参数不需要包含在 URI 模板中。当客户端请求资源时，FastMCP 将：

- 从 URI 中提取模板中包含的参数的参数值
- 对 URI 模板中不存在的任何函数参数使用默认值

这允许灵活的 API 设计。例如，一个带有可选参数的简单搜索模板：

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")

@mcp.resource("search://{query}")
def search_resources(query: str, max_results: int = 10, include_archived: bool = False) -> dict:
    """Search for resources matching the query string."""
    # Only 'query' is required in the URI, the other parameters use their defaults
    results = perform_search(query, limit=max_results, archived=include_archived)
    return {
        "query": query,
        "max_results": max_results,
        "include_archived": include_archived,
        "results": results
    }

使用此模板，客户端可以请求 `search://python`，函数将以 `query="python"`、`max_results=10`、`include_archived=False` 的方式调用。MCP 开发人员仍可使用更具体的参数直接调用底层的 `search_resources` 函数。

一个更强大的模式是使用多个 URI 模板注册单个函数，允许以不同的方式访问相同的数据：

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")

# Define a user lookup function that can be accessed by different identifiers
@mcp.resource("users://email/{email}")
@mcp.resource("users://name/{name}")
def lookup_user(name: str | None = None, email: str | None = None) -> dict:
    """Look up a user by either name or email."""
    if email:
        return find_user_by_email(email) # pseudocode
    elif name:
        return find_user_by_name(name) # pseudocode
    else:
        return {"error": "No lookup parameters provided"}

现在，LLM 或客户端可以通过两种不同的方式检索用户信息：

- `users://email/alice@example.com`→ 通过电子邮件查找用户（名称=无）
- `users://name/Bob`→ 按姓名查找用户（电子邮件=无）

在这个堆叠装饰器模式中：

- 该参数仅在使用模板name时提供`users://name/{name}`
- 该参数仅在使用模板email时提供`users://email/{email}`
- 当 URI 中未包含每个参数时，默认值为None
- 函数逻辑处理提供的任何参数

模板提供了一种强大的方法来公开遵循类似 REST 原则的参数化数据访问点。

## 错误处理
如果您的资源函数遇到错误，您可以引发标准 Python 异常（ValueError\TypeError\FileNotFoundError等）或 FastMCP ResourceError。

出于安全考虑，大多数异常ResourceError在发送给客户端之前都会被封装在一个泛型中，并屏蔽内部错误详细信息。但是，如果您ResourceError直接引发异常，其内容将包含在响应中。这允许您根据客户端的需要，提供信息丰富的错误消息。

In [ ]:
from fastmcp import FastMCP
from fastmcp.exceptions import ResourceError

mcp = FastMCP(name="DataServer")

@mcp.resource("resource://safe-error")
def fail_with_details() -> str:
    """This resource provides detailed error information."""
    # ResourceError contents are sent back to clients
    raise ResourceError("Unable to retrieve data: file not found")

@mcp.resource("resource://masked-error")
def fail_with_masked_details() -> str:
    """This resource masks internal error details."""
    # Other exceptions are converted to ResourceError with generic message
    raise ValueError("Sensitive internal file path: /etc/secrets.conf")

@mcp.resource("data://{id}")
def get_data_by_id(id: str) -> dict:
    """Template resources also support the same error handling pattern."""
    if id == "secure":
        raise ValueError("Cannot access secure data")
    elif id == "missing":
        raise ResourceError("Data ID 'missing' not found in database")
    return {"id": id, "value": "data"}

## 服务器行为
​
### 重复资源
您可以配置 FastMCP 服务器如何处理尝试使用同一 URI 注册多个资源或模板的情况。请在FastMCP初始化期间使用此设置`on_duplicate_resources`

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(
    name="ResourceServer",
    on_duplicate_resources="error" # Raise error on duplicates
)

@mcp.resource("data://config")
def get_config_v1(): return {"version": 1}

# This registration attempt will raise a ValueError because
# "data://config" is already registered and the behavior is "error".
# @mcp.resource("data://config")
# def get_config_v2(): return {"version": 2}

重复行为选项包括：

- "warn"（默认）：记录警告，新的资源/模板将替换旧的资源/模板。
- "error"：提出ValueError，防止重复注册。
- "replace"：默默地用新的资源/模板替换现有的资源/模板。
- "ignore"：保留原始资源/模板并忽略新的注册尝试。